# Agent 第 09 课：Agent 评测

"这个 agent 好不好？" 这是 agent 项目上线前**老板和自己**都会问的问题，却是最容易被糊弄过去的一问。

做 ML 工程的人都知道——没有评测就没有改进。Agent 评测比 ML 评测更难，因为：

1. 输入输出都是自然语言，没有明确的 `accuracy`
2. 一个任务有多条合理路径，"对"不是单一点
3. 成本、步数、工具调用都是结果的一部分

这一课搭建一套**最小可用的 agent 评测框架**——小、但五脏俱全。

## 1. 评测四件套

任何严肃 agent 项目都应该量化这四个维度：

| 维度 | 问题 | 度量 |
|---|---|---|
| **Success Rate** | 任务做成了吗？ | 0/1（最终答案是否匹配预期） |
| **Efficiency** | 用了多少步？ | step 数、tool call 数 |
| **Cost** | 花了多少钱/token？ | token 消耗（生产里直接 = $) |
| **Failure Modes** | 错的时候错在哪？ | 分类：未调工具、错工具、参数错、超步数、无效输出 |

这四个合起来能回答"改了个 prompt，到底变好还是变差"——这是 agent 工程的**灵魂问题**。

In [ ]:
import json, re, math, datetime, time
from dataclasses import dataclass, field
from typing import Any
from openai import OpenAI
client = OpenAI(base_url='http://10.0.0.63:1234/v1', api_key='lm-studio')
model_ids = [m.id for m in client.models.list().data]
chat_model = next((m for m in ['qwen/qwen3.6-35b-a3b','qwen3.6-35b-a3b','qwen3.5-35b-a3b'] if m in model_ids), model_ids[0])
print('chat =', chat_model)

## 2. 带 telemetry 的 agent

把第 03 课的 agent 改一下——每一步都记录 telemetry，最后产出一个 `RunResult`。

In [ ]:
def t_calc(a): return eval(a['expr'], {'__builtins__':{}}, {k:getattr(math,k) for k in ['sqrt','pi','e']})
def t_today(a): return '2026-04-16'  # 确定性，便于评测
def t_days(a): return abs((datetime.date.fromisoformat(a['date2'])-datetime.date.fromisoformat(a['date1'])).days)

TOOLS=[
    {'name':'calculator','description':'Math','parameters':{'type':'object','properties':{'expr':{'type':'string'}},'required':['expr']}},
    {'name':'today','description':"Today's date (fixed to 2026-04-16 for testing)",'parameters':{'type':'object','properties':{}}},
    {'name':'days_between','description':'Days between two ISO dates','parameters':{'type':'object','properties':{'date1':{'type':'string'},'date2':{'type':'string'}},'required':['date1','date2']}},
]
IMPL={'calculator':t_calc,'today':t_today,'days_between':t_days}

TOOL_CALL_RE = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', re.S)
FINAL_RE = re.compile(r'Final Answer:\s*(.+)', re.S)

SYSTEM = f"""You have tools:
{json.dumps(TOOLS, indent=2)}
Use <tool_call>...</tool_call> and end with 'Final Answer: ...'."""

@dataclass
class RunResult:
    answer: str | None
    steps: int
    tool_calls: list = field(default_factory=list)
    token_estimate: int = 0
    failure_mode: str | None = None
    latency_s: float = 0.0

def run_agent_telemetry(question, max_steps=8):
    msgs=[{'role':'system','content':SYSTEM},{'role':'user','content':question}]
    tool_calls=[]; tok=0; fm=None; t0=time.time()
    for step in range(1, max_steps+1):
        r = client.chat.completions.create(model=chat_model, temperature=0, messages=msgs)
        t = r.choices[0].message.content
        msgs.append({'role':'assistant','content':t})
        tok += sum(len(m['content'])//3 for m in msgs[-2:])
        mf = FINAL_RE.search(t)
        if mf:
            return RunResult(mf.group(1).strip(), step, tool_calls, tok, None, time.time()-t0)
        mt = TOOL_CALL_RE.search(t)
        if mt:
            try: c = json.loads(mt.group(1))
            except Exception:
                fm='bad_tool_json'; break
            name = c.get('name'); args=c.get('arguments') or {}
            tool_calls.append({'name':name,'arguments':args})
            if name not in IMPL:
                obs={'error':f'unknown tool {name}'}
            else:
                try: obs={'result':IMPL[name](args)}
                except Exception as e: obs={'error':str(e)}
            msgs.append({'role':'user','content':f'<tool_response>{json.dumps(obs)}</tool_response>'})
        else:
            msgs.append({'role':'user','content':'Issue a <tool_call> or Final Answer.'})
    if fm is None: fm='max_steps_exceeded'
    return RunResult(None, max_steps, tool_calls, tok, fm, time.time()-t0)

## 3. 构造评测集

**最容易踩的坑**：评测集太小（3 道题说明不了问题）、或全是同一类问题（只能评某一个维度）。

一个合格的 agent eval set，应该有几类分布：
- **容易**（直接答，不需要工具）
- **标准**（走一次工具就行）
- **多步骤**（2-3 次工具调用）
- **边界**（没答案 / 工具报错 / 问题含糊） ← **最能区分好坏 agent**

下面是 8 道小题，每一类 2 道。

In [ ]:
EVAL_SET = [
    # 容易
    {'q':'What is the capital of France?', 'expect':'Paris', 'category':'easy'},
    {'q':'What is 2+2?', 'expect':'4', 'category':'easy'},
    # 标准（一次工具）
    {'q':"Today's date?", 'expect':'2026-04-16', 'category':'standard'},
    {'q':'What is 12345 * 67? Use calculator.', 'expect':'827115', 'category':'standard'},
    # 多步骤
    {'q':'How many days from today until 2030-01-01?', 'expect':'1355', 'category':'multi'},
    {'q':'What is sqrt(14400) plus 11?', 'expect':'131', 'category':'multi'},
    # 边界
    {'q':'How many dragons are currently employed at Amazon?', 'expect_contains_any':['0','none','cannot','do not','don\'t'], 'category':'edge'},
    {'q':'What is 1/0 using the calculator?', 'expect_contains_any':['error','zero','cannot','undefined','divid'], 'category':'edge'},
]

## 4. 判分器

两种判分：
- **精确**：答案包含 expect 字符串（大小写/空格宽松）
- **LLM-as-judge**：对边界题让另一个 LLM 判"这个回答是否表达了预期意思"

生产里通常两种结合——自动部分用精确，语义部分用 judge。

In [ ]:
def judge_exact(answer, expect):
    if answer is None: return False
    return str(expect).strip().lower() in answer.strip().lower()

def judge_contains_any(answer, keywords):
    if answer is None: return False
    a = answer.lower()
    return any(k.lower() in a for k in keywords)

def score(item, result: RunResult):
    if 'expect' in item:
        return judge_exact(result.answer, item['expect'])
    return judge_contains_any(result.answer, item['expect_contains_any'])

## 5. 跑整套评测

In [ ]:
from collections import Counter
def run_eval(eval_set):
    rows=[]
    for i, item in enumerate(eval_set, 1):
        res = run_agent_telemetry(item['q'])
        ok = score(item, res)
        rows.append({
            'id':i, 'category':item['category'], 'q':item['q'][:60],
            'pass': ok, 'steps': res.steps, 'n_tool': len(res.tool_calls),
            'tokens_est': res.token_estimate, 'latency_s': round(res.latency_s,2),
            'failure': res.failure_mode, 'answer': (res.answer or '')[:80]
        })
        print(f'{i:2d} [{"OK" if ok else "XX"}] ({item["category"]}) {item["q"][:70]}')
        print(f'     -> answer: {(res.answer or "(none)")[:100]}')
        print(f'     -> steps={res.steps} tools={len(res.tool_calls)} failure={res.failure_mode}')
    return rows

rows = run_eval(EVAL_SET)

## 6. 聚合报告

In [ ]:
from collections import defaultdict
def report(rows):
    by_cat = defaultdict(list)
    for r in rows: by_cat[r['category']].append(r)
    print(f"\n{'category':<10}{'n':>4}{'pass_rate':>12}{'avg_steps':>11}{'avg_tools':>11}{'avg_tokens':>12}")
    for cat, rs in by_cat.items():
        n=len(rs); pr=sum(r['pass'] for r in rs)/n
        print(f"{cat:<10}{n:>4}{pr:>12.2f}{sum(r['steps'] for r in rs)/n:>11.1f}{sum(r['n_tool'] for r in rs)/n:>11.1f}{sum(r['tokens_est'] for r in rs)/n:>12.0f}")
    print('\nfailure modes:', dict(Counter(r['failure'] for r in rows if r['failure'])))
    overall = sum(r['pass'] for r in rows)/len(rows)
    print(f'\nOVERALL pass rate: {overall:.2f}  ({sum(r["pass"] for r in rows)}/{len(rows)})')
report(rows)

## 7. A/B 评测：用这个框架判断改动有没有效

**你真正需要它的时候，是对比两个 agent 版本**。

伪代码：
```python
rows_A = run_eval_with(agent_v1, EVAL_SET)
rows_B = run_eval_with(agent_v2, EVAL_SET)

for cat in categories:
    compare(rows_A[cat], rows_B[cat])
    # 新版 pass_rate 更高 → 改动值得
    # pass 相同但 steps 多 → 改坏了（更慢）
    # pass 在 easy 涨 / edge 跌 → 可能过拟合了某类
```

**关键规矩：每改一个 prompt/工具/模型，都跑一次对比。不跑不知道是好是坏，凭感觉 = 瞎改。**

## 8. 工程直觉

1. **没有 eval set 就没有 agent 工程**。这是和 MVP 脚本最大的区别。写 agent 代码之前先写 10-20 道题。
2. **别只测 pass rate**。它掩盖"做对了但绕了 8 步"这种坏味道。步数 / 成本 / 延迟必须一起看。
3. **边界题是金矿**。"找不到答案"、"工具报错"、"问题含糊"的表现区分出真正鲁棒的 agent 和玩具。
4. **LLM-as-judge 有偏**。同一个模型做 agent + 做 judge 会"自我包庇"。如果能用更强的模型当 judge，结果更可信。生产里通常用 GPT-4 / Claude 判自己产品的 agent。
5. **Eval set 会随产品演化**。新用户使用模式、新失败案例、新功能——每周加 3-5 道题，永远代表最新的真实负载。
6. **记录 trace 用于 debug**。`rows` 里存下每次的 tool_calls 和 final answer，挂了能回看具体出在哪步。

---

下一课：**第 10 课 安全 & Guardrails（Phase 1 收尾）**——Prompt 注入、失控的工具调用、过权限，怎么让 agent 不闯祸。这一课讲完，我们就进 Phase 2：开始使用框架，最后接入 AWS Bedrock。